In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, TimestampType
from pyspark.sql.functions import trim, col

In [0]:
df = spark.table("workspace.bronze.orders")

In [0]:
RENAME_MAP = {
    "customer_id": "customer_order_key"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
timestamp_cols = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]

for c in timestamp_cols:
    df = df.withColumn(c, F.to_timestamp(F.col(c)))

In [0]:
df = df.withColumn("order_status", F.lower(col("order_status")))

In [0]:
df = df.withColumn(
    "delivery_delay_days",
    F.when(
        col("order_delivered_customer_date").isNotNull() & 
        col("order_estimated_delivery_date").isNotNull(),
        F.datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))
    ).otherwise(None)
)

df = df.withColumn(
    "is_late",
    F.when(col("delivery_delay_days") > 0, True)
     .when(col("delivery_delay_days").isNotNull(), False)
     .otherwise(None)
)

In [0]:
df = df.filter(col("order_id").isNotNull() & col("customer_order_key").isNotNull())

In [0]:
df = df.dropDuplicates(["order_id"])

In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())

In [0]:
# Check duplicates
df.groupBy("order_id").count().filter("count > 1").display()

# Check nulls
df.filter(col("order_id").isNull() | col("customer_order_key").isNull()).display()

In [0]:
df.show(5)

In [0]:
df = df.withColumn("order_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))

df.write.partitionBy("order_month").mode("overwrite").format("delta").saveAsTable("workspace.silver.orders")